In [ ]:
# @title Upload from Shopify
import pandas as pd
import numpy as np
import io
import warnings
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files

from prophet import Prophet
import xgboost as xgb

warnings.filterwarnings("ignore")

# 1. Define the dynamic Shopify link
# This link uses the ShopifyQL. You can add your account name which should be an Admin in the {Brackets} in the line below
# shopify_url = "https://admin.shopify.com/store/{your_account}/analytics/reports/262537484?ql=FROM+sales%0A++SHOW+net_sales%0A++GROUP+BY+day%0A++SINCE+2023-01-01+UNTIL+yesterday%0A++ORDER+BY+day+ASC"

display(HTML(f"""
<div style="padding:15px; border:2px solid #9575CD; border-radius:10px; background-color:#f9f7ff; font-family:sans-serif;">
  <h3 style="color:#5c429c; margin-top:0;">🚀 Step 1: Download Your Data</h3>
  <p>Click the link below to open your Shopify report. Once it opens, click <b>Export</b> to download the CSV.</p>
  <a href="{shopify_url}" target="_blank" style="background-color:#9575CD; color:white; padding:10px 20px; text-decoration:none; border-radius:5px; font-weight:bold; display:inline-block;">Open Shopify Report</a>
  <p style="font-size:0.8em; color:#666; margin-top:10px;">Note: You must be logged into your Shopify Admin.</p>
</div>
<br>
"""))

print("--- 📥 Step 2: Upload the CSV ---")
uploaded = files.upload()

for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]))

    # Auto-mapping columns for the forecast
    date_col = next((c for c in df.columns if 'day' in c.lower() or 'date' in c.lower()), df.columns[0])
    sales_col = next((c for c in df.columns if 'sales' in c.lower() or 'total' in c.lower()), df.columns[1])

    forecast_df = df[[date_col, sales_col]].copy()
    forecast_df.columns = ['ds', 'y']
    forecast_df['ds'] = pd.to_datetime(forecast_df['ds'])
    forecast_df = forecast_df.dropna().sort_values('ds')

    print(f"\n✅ Data Ready! {len(forecast_df)} days loaded.")

In [ ]:
# @title Past Campaigns

# 1. Define the Campaign list (Start and End dates inclusive)

campaign_data = [
    ("Father's Day Tech Box 30€", "2025-05-23", "2025-05-26"),
    ("Father's Day 10% Off", "2025-05-28", "2025-05-29"),
    ("Tech Giveaway Sweepstakes", "2025-05-31", "2025-06-02"),
    ("Mini Mystery Gadget Box", "2025-06-06", "2025-06-08"),
    ("Pentecost Promo Box 30€", "2025-06-08", "2025-06-09"),
    ("VIP Mail 10% Discount", "2025-06-25", "2025-06-26"),
    ("1 Cent Smart LED Light", "2025-06-26", "2025-06-30"),
    ("Electronics Flash Sales", "2025-06-25", "2025-07-01"),
    ("Buy 1 Get 1 USB Cables", "2025-07-09", "2025-07-11"),
    ("1 Cent Screen Protector", "2025-07-14", "2025-07-16"),
    ("10% Off Smart Home Gear", "2025-07-25", "2025-07-27"),
    ("70k Milestone Tech Box (70€)", "2025-07-26", "2025-07-28"),
    ("1€ Accessory Sale", "2025-07-30", "2025-08-03"),
    ("Discord VIP + 10% Discount", "2025-08-04", "2025-08-05"),
    ("8.8 Super Tech Sets", "2025-08-08", "2025-08-10"),
    ("1 Cent Bluetooth Headphones", "2025-08-13", "2025-08-17"),
    ("80k Milestone Scratch Card", "2025-08-20", "2025-08-22"),
    ("1 Cent Campaign Smart Plug", "2025-08-28", "2025-08-31"),
    ("1€ Accessory Sale", "2025-09-01", "2025-09-05"),
    ("1 Cent Power Bank", "2025-10-18", "2025-10-20"),
    ("Halloween 1 Cent Gadget", "2025-10-29", "2025-10-31"),
    ("iPhone Giveaway", "2025-11-03", "2025-11-10"),
    ("Singles Day Electronics", "2025-11-11", "2025-11-13"),
    ("One Cent Phone Case", "2026-01-28", "2026-01-30"),
    ("Tech New Year Launch", "2026-02-14", "2026-02-14"),
    ("Tech New Year Launch", "2026-02-16", "2026-02-16"),
    ("Tech New Year Launch", "2026-02-17", "2026-02-17"),
    ("One Cent Fast Charger", "2026-02-25", "2026-02-25"),
    ("One Cent Fast Charger", "2026-02-26", "2026-02-28")
]

# 2. Reset promotion columns to ensure a fresh start
forecast_df['promotion'] = 0
forecast_df['campaign_name'] = "No Campaign"

# 3. Apply the date-range mask
for name, start, end in campaign_data:
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)

    # Identify all rows where 'ds' is between (and including) start and end
    mask = (forecast_df['ds'] >= start_dt) & (forecast_df['ds'] <= end_dt)

    forecast_df.loc[mask, 'promotion'] = 1
    # We overwrite or append the campaign name
    forecast_df.loc[mask, 'campaign_name'] = name

# 4. Validation Print
promo_days = forecast_df[forecast_df['promotion'] == 1]
print(f"✅ Mapping Complete.")
print(f"Total days in dataset: {len(forecast_df)}")
print(f"Total active campaign days: {len(promo_days)}")

In [ ]:
#@title ⚙️ Training ML Models (30-Day Stability Anchor)

# 1. Data Pools remain the same
cutoff_long = forecast_df['ds'].max() - pd.Timedelta(days=730)
df_prophet_train = forecast_df[forecast_df['ds'] >= cutoff_long].copy()
cutoff_short = forecast_df['ds'].max() - pd.Timedelta(days=365)
df_xgb_train = forecast_df[forecast_df['ds'] >= cutoff_short].copy()



# --- MODEL 1: PROPHET (Strategic Base) ---
df_p = df_prophet_train.copy()
df_p['y'] = np.log1p(df_p['y'])
m = Prophet(interval_width=0.60, daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True, seasonality_mode='multiplicative')
if 'promotion' in df_p.columns: m.add_regressor('promotion')
m.fit(df_p)
future_base = m.make_future_dataframe(periods=90)
if 'promotion' in df_p.columns: future_base['promotion'] = 0
forecast_base = m.predict(future_base)
for col in ['yhat', 'yhat_lower', 'yhat_upper']:
    forecast_base[col] = np.expm1(forecast_base[col])
forecast_base = forecast_base.rename(columns={'yhat': 'yhat_prophet'})

# --- MODEL 2: XGBOOST (30-DAY STABILITY LOGIC) ---
df_x = df_xgb_train.copy()
df_x['day_of_week'] = df_x['ds'].dt.dayofweek
df_x['month'] = df_x['ds'].dt.month
df_x['lag_1'] = df_x['y'].shift(1)
df_x['lag_7'] = df_x['y'].shift(7)
df_x['lag_30'] = df_x['y'].shift(30) # New: Monthly Lag
df_x['rolling_30'] = df_x['y'].rolling(window=30).mean() # New: Monthly Average
df_x = df_x.dropna()

features = ['day_of_week', 'month', 'promotion', 'lag_1', 'lag_7', 'lag_30', 'rolling_30']
xgb_mod = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.03) # Slower learning rate = more stability
xgb_mod.fit(df_x[features], df_x['y'])

# Projecting with 30-Day Context
future_xgb = forecast_base[['ds', 'promotion']].copy()
future_xgb['day_of_week'] = future_xgb['ds'].dt.dayofweek
future_xgb['month'] = future_xgb['ds'].dt.month
future_xgb['lag_1'] = df_xgb_train['y'].iloc[-1]
future_xgb['lag_7'] = df_xgb_train['y'].iloc[-7]
future_xgb['lag_30'] = df_xgb_train['y'].iloc[-30]
future_xgb['rolling_30'] = df_xgb_train['y'].tail(30).mean() # The "Anchor"

forecast_base['yhat_xgb'] = xgb_mod.predict(future_xgb[features])
forecast_base['yhat_ml'] = (forecast_base['yhat_prophet'] + forecast_base['yhat_xgb']) / 2



In [ ]:
#@title 🗓️ MARKETING CALENDAR (STORAGE)
# Update this list with your campaigns.

CAMPAIGN_STORAGE = {
    # 'YYYY-MM-DD': Lift Percentage (e.g., 10 = +10% revenue)

    '2026-03-01': 10,
    '2026-03-07': 10,
    '2026-03-08': 10,
    '2026-03-11': 15,
    '2026-03-12': 10,
    '2026-03-13': 10,
    '2026-03-14': 10,
    '2026-03-15': 10,
    '2026-03-25': 10,
    '2026-03-26': 10,
    '2026-03-27': 10,
    '2026-03-28': 10,
    '2026-03-29': 10

}

# --- Internal Update: Sync with the Forecast Model ---
for d_str in CAMPAIGN_STORAGE.keys():
    try:
        d_dt = pd.to_datetime(d_str)
        mask = (forecast_base['ds'] == d_dt)
        if mask.any():
            forecast_base.loc[mask, 'promotion'] = 1
    except:
        continue

print(f"✅ Loaded {len(CAMPAIGN_STORAGE)} campaign days into memory.")

In [ ]:
# @title 🚀 STRATEGIC COMMANDER (CLEAN VIEW) { display-mode: "form" }


# --- UI SETUP ---
growth_slider = widgets.IntSlider(value=5, min=0, max=50, step=1, description='Challenge %')
view_slider = widgets.IntSlider(value=14, min=7, max=90, step=1, description='Forecast Days')
history_slider = widgets.IntSlider(value=21, min=7, max=180, step=1, description='History Days')

quick_input = widgets.Text(
    value='',
    placeholder='YYYY-MM-DD:Lift',
    description='Campaigns:',
    continuous_update=False
)

print_btn = widgets.Button(description="Print Report", icon="print", style={'button_color': '#2c3e50'})

def on_print_clicked(b):
    display(HTML("<script>window.print();</script>"))
print_btn.on_click(on_print_clicked)

ui = widgets.VBox([
    widgets.HBox([growth_slider, view_slider, history_slider, print_btn]),
    widgets.HBox([quick_input, widgets.Label("(Enter YYYY-MM-DD:Lift & press Enter to simulate manual campaign boosts)")])
], layout={'border': '1px solid #ddd', 'padding': '15px', 'background-color': '#fcfcfc'})
out = widgets.Output()

def update_live_view(growth, quick_str, days, history):
    with out:
        clear_output(wait=True)
        if 'forecast_base' not in globals() or 'forecast_df' not in globals():
            print("❌ DATA NOT FOUND: Please run the training cells above.")
            return

        # 1. SETUP DATA
        f_active = forecast_base.copy()
        last_dt = forecast_df['ds'].max()
        target_start = last_dt.replace(day=1)
        target_end = (target_start + pd.Timedelta(days=32)).replace(day=1) - pd.Timedelta(days=1)

        # Comparison
        prev_end = target_start - pd.Timedelta(days=1)
        prev_start = prev_end.replace(day=1)
        actual_prev_month = forecast_df[(forecast_df['ds'] >= prev_start) & (forecast_df['ds'] <= prev_end)]['y'].sum()

        # 2. APPLY CHALLENGE & QUICK CAMPAIGNS
        challenge_mult = 1 + (growth / 100)

        temp_storage = {}
        if quick_str:
            try:
                for item in quick_str.split(','):
                    d, l = item.split(':')
                    temp_storage[d.strip()] = 1 + (float(l)/100)
            except:
                pass

        current_event_map = {k: 1 + (v/100) for k, v in CAMPAIGN_STORAGE.items()} if 'CAMPAIGN_STORAGE' in globals() else {}
        current_event_map.update(temp_storage)

        def apply_boost(row):
            date_key = row['ds'].strftime('%Y-%m-%d')
            is_target_month = (pd.to_datetime(row['ds']) >= target_start) and (pd.to_datetime(row['ds']) <= target_end)
            active_uplift = challenge_mult if is_target_month else 1.0
            ev_mult = current_event_map.get(date_key, 1.0)

            # Main Line: Adjusted Prophet (Strategic)
            row['yhat_p_adj'] = row['yhat_prophet'] * active_uplift * ev_mult

            # Purple Line: ML Ensemble (Prophet + XGBoost blended)
            # We use 'yhat_ml' which was already calculated in your training cell
            row['yhat_ensemble_adj'] = row['yhat_ml'] * active_uplift * ev_mult

            row['yhat_up_adj'] = row['yhat_upper'] * active_uplift * ev_mult
            row['yhat_low_adj'] = row['yhat_lower'] * active_uplift * ev_mult
            return row
        f_active = f_active.apply(apply_boost, axis=1)

        # 3. STATS
        actuals_mtd = forecast_df[(forecast_df['ds'] >= target_start) & (forecast_df['ds'] <= last_dt)]['y'].sum()
        est_long_p = actuals_mtd + f_active[(f_active['ds'] > last_dt) & (f_active['ds'] <= target_end)]['yhat_p_adj'].sum()
        projected_total = est_long_p
        ceo_goal = f_active[(f_active['ds'] >= target_start) & (f_active['ds'] <= target_end)]['yhat_p_adj'].sum()

        days_passed = (last_dt - target_start).days + 1
        total_days = (target_end - target_start).days + 1
        percent_time, percent_goal = (days_passed/total_days)*100, (actuals_mtd/ceo_goal)*100
        diff_val = projected_total - actual_prev_month
        diff_pct = (diff_val / actual_prev_month) * 100 if actual_prev_month > 0 else 0

        # 4. DIAGRAM
        fig, ax = plt.subplots(figsize=(25, 6.5))
        plt.subplots_adjust(top=0.75)

        history_start = last_dt - pd.Timedelta(days=history)
        ax.plot(forecast_df[forecast_df['ds'] >= history_start]['ds'],
                forecast_df[forecast_df['ds'] >= history_start]['y'],
                color='#2c3e50', label='Actual Sales', alpha=0.3, linewidth=3)

        view_start, view_end = history_start, last_dt + pd.Timedelta(days=days)

        # Forecast restricted to max 21 days back
        forecast_vis_start = last_dt - pd.Timedelta(days=21)
        fore_plot = f_active[(f_active['ds'] >= forecast_vis_start) & (f_active['ds'] <= view_end)]

        ax.fill_between(fore_plot['ds'], fore_plot['yhat_low_adj'], fore_plot['yhat_up_adj'], color='#2980B9', alpha=0.08, label='60% Confidence Band')

        for date_str, mult in current_event_map.items():
            dt = pd.to_datetime(date_str)
            if view_start <= dt <= view_end:
                ax.axvspan(dt - pd.Timedelta(hours=12), dt + pd.Timedelta(hours=12), color='#e74c3c', alpha=0.1)

        # PLOT PROPHET (Strategic Target)
        ax.plot(fore_plot['ds'],
                fore_plot['yhat_p_adj'],
                color='#2980B9', label='Strategic Target (Prophet)', linewidth=2.5)

        # PLOT ENSEMBLE (Purple Dotted Line)
        ax.plot(fore_plot['ds'],
                fore_plot['yhat_ensemble_adj'],
                color='purple', label='Ensemble Forecast (Prophet+XGB)', linestyle=':', linewidth=1.2, alpha=0.8)

        ax.axvline(last_dt, color='#2c3e50', linestyle=':', alpha=0.5)

        dots_df = forecast_df.merge(f_active[['ds', 'yhat_p_adj']], on='ds').tail(14)
        ax.scatter(dots_df[dots_df['y'] >= dots_df['yhat_p_adj']]['ds'], dots_df[dots_df['y'] >= dots_df['yhat_p_adj']]['y'], color='#27ae60', s=100, edgecolors='white', zorder=6, label='Hit Challenge')
        ax.scatter(dots_df[dots_df['y'] < dots_df['yhat_p_adj']]['ds'], dots_df[dots_df['y'] < dots_df['yhat_p_adj']]['y'], color='#e74c3c', s=100, edgecolors='white', zorder=6, label='Missed Challenge')

        # Formatting
        ax.set_xlim(view_start, view_end)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(1000))
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('€%.0f'))
        ax.xaxis.set_major_locator(mdates.DayLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
        plt.xticks(rotation=90)
        sns.despine(left=True, bottom=True)
        ax.grid(True, linestyle=':', alpha=0.3)

        plt.text(0.0, 1.3, f"{target_start.strftime('%B %Y').upper()} NET SALES", transform=ax.transAxes, fontsize=14, fontweight='bold', color='#34495e')
        ax.legend(bbox_to_anchor=(1.0, 1.38), loc='upper right', frameon=False, ncol=3, fontsize=8)
        plt.text(0.0, 1.20, f"ESTIMATE: EUR {projected_total:,.0f}", transform=ax.transAxes, fontsize=20, fontweight='bold', color='#2c3e50')

        comp_color = "#27ae60" if diff_val >= 0 else "#e74c3c"
        plt.text(0.0, 1.12, f"vs. PREV MONTH: {'+' if diff_val > 0 else ''}{diff_val:,.0f} ({diff_pct:+.1f}%)", transform=ax.transAxes, fontsize=12, fontweight='bold', color=comp_color)
        rr_color = "#27ae60" if percent_goal >= percent_time else "#e74c3c"
        plt.text(0.0, 1.07, f"RUN RATE: {percent_goal:.1f}% OF GOAL ({percent_time:.1f}% OF MONTH)", transform=ax.transAxes, fontsize=12, fontweight='bold', color=rr_color)

        plt.show()

        # --- 5. ENHANCED TABLE ---
        past_view = forecast_df.merge(f_active[['ds', 'yhat_p_adj']], on='ds').tail(14)
        future_view = f_active[f_active['ds'] > last_dt].head(7).copy()
        future_view['y'] = np.nan

        combined_view = pd.concat([past_view, future_view])
        combined_view['Diff'] = combined_view['y'] - combined_view['yhat_p_adj']
        combined_view['Date'] = combined_view['ds'].dt.strftime('%d %b')

        t_final = combined_view.set_index('Date')[['y', 'yhat_p_adj', 'Diff']].T
        t_final.index = ['Actual Net Sales', 'Target', 'Variance']

        sep_idx = 13

        display(t_final.style.apply(lambda x: [
            "color: #e74c3c; font-weight: bold" if (not pd.isna(val) and val < 0 and x.name == 'Variance')
            else ("color: #27ae60; font-weight: bold" if (not pd.isna(val) and val >= 0 and x.name == 'Variance')
            else "color: #95a5a6; font-style: italic" if (pd.isna(val))
            else "") for val in x], axis=1)
            .set_properties(subset=t_final.columns[sep_idx:sep_idx+1], **{'border-right': '3px solid #2c3e50'})
            .format("€{:,.0f}", na_rep="-")
            .set_table_attributes('style="width:100%; text-align: center; border-collapse: collapse;"'))

# Init
interactive_out = widgets.interactive_output(update_live_view, {
    'growth': growth_slider,
    'quick_str': quick_input,
    'days': view_slider,
    'history': history_slider
})
display(ui, out)